# Hybrid CNN + Transformer Network

This idea combines *CNNs* (good at local features) with *Transformers* (good at global relationships).
The *CNN* extracts spatially local features efficiently, while the *Transformer* encoder captures long-range dependencies between image regions. This hybrid design improves representation learning without excessive computational cost.

## Dataset: CIFAR-10

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

In [ ]:
def load_cifar10_data(data_dir):
    """Load all CIFAR-10 training data"""
    X_train = []
    y_train = []
    
    # Load training batches
    for i in range(1, 6):
        batch = unpickle(f"{data_dir}/data_batch_{i}")
        X_train.append(batch[b'data'])
        y_train.append(batch[b'labels'])
    
    # Concatenate all batches
    X_train = np.concatenate(X_train)
    y_train = np.concatenate(y_train)
    
    # Load test batch
    test_batch = unpickle(f"{data_dir}/test_batch")
    X_test = test_batch[b'data']
    y_test = np.array(test_batch[b'labels'])
    
    return X_train, y_train, X_test, y_test

In [ ]:
# Load the data
data_dir = "/Users/somchannreaksmey/Documents/I4-AMS-B/Artificial-Intelligence/cnn/cifar-10-batches-py"
X_train, y_train, X_test, y_test = load_cifar10_data(data_dir)

# Reshape data: CIFAR-10 images are 32x32 with 3 color channels (RGB)
X_train = X_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # Shape: (50000, 32, 32, 3)
X_test = X_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)    # Shape: (10000, 32, 32, 3)

# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

In [ ]:
# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Visualize some samples
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i])
    plt.xlabel(class_names[y_train[i]])
plt.show()

## Network Architecture

In [ ]:
import os
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)

set_seed()

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

print("Using device:", device)


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),  # downsample by 2
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Pre-LN Transformer encoder block."""

    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, N, D]
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(attn_out)

        h = self.ln2(x)
        x = x + self.mlp(h)
        return x

In [ ]:
class CNNTransformer(nn.Module):
    def __init__(self, num_classes=10, dim=128, depth=3, heads=4):
        super().__init__()

        self.cnn = nn.Sequential(
            ConvBlock(3, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256)
        )

        self.patch_proj = nn.Linear(256 * 2 * 2, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, 4, dim))

        self.transformer = nn.Sequential(
            *[TransformerBlock(dim, heads) for _ in range(depth)]
        )

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)                  # [B, 256, 4, 4]
        x = x.unfold(2, 2, 2).unfold(3, 2, 2)
        x = x.contiguous().view(x.size(0), 4, -1)
        x = self.patch_proj(x) + self.pos_embed
        x = self.transformer(x)
        x = self.norm(x).mean(dim=1)
        return self.head(x)


In [ ]:
train_tfms = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2470, 0.2435, 0.2616))
])

test_tfms = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465),
                (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_tfms
)

test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_tfms
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)


In [ ]:
def accuracy(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()

def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(out, y)

    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, loss_fn):
    model.eval()
    total_loss, total_acc = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = loss_fn(out, y)

        total_loss += loss.item()
        total_acc += accuracy(out, y)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

In [ ]:
epochs = 30
best_acc = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, loss_fn
    )
    val_loss, val_acc = eval_epoch(
        model, test_loader, loss_fn
    )

    # Save history
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")


In [ ]:
plt.figure()
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()